In [1]:
#que decido en la fase 3
import pandas as pd
from pathlib import Path

ruta_eda = Path(r'C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\04_eda')
ruta_proc = Path(r'C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed')

df_eda = pd.read_parquet(ruta_eda / 'df_eda_final.parquet')
df_alumno = pd.read_parquet(ruta_proc / 'df_alumno.parquet')

# Alumnos con 2+ titulaciones en df_alumno
n_tit = df_alumno.groupby('per_id_ficticio')['titulacion'].nunique()
multiples = n_tit[n_tit > 1].index.tolist()

print(f"Alumnos con 2+ titulaciones en df_alumno: {len(multiples)}")

# Ver 5 ejemplos concretos
for pid in multiples[:5]:
    filas = df_alumno[df_alumno['per_id_ficticio'] == pid][
        ['per_id_ficticio','titulacion','curso_aca_ini','egresado']
    ].sort_values('curso_aca_ini')
    print(f"\nAlumno {pid}:")
    print(filas.to_string(index=False))

Alumnos con 2+ titulaciones en df_alumno: 2593

Alumno 3672:
 per_id_ficticio                                                         titulacion  curso_aca_ini egresado
            3672 Grado en Ingeniería en Diseño Industrial y Desarrollo de Productos           2014        N
            3672                                      Grado en Arquitectura Técnica           2015        N
            3672                                      Grado en Arquitectura Técnica           2015        N

Alumno 4464:
 per_id_ficticio                                titulacion  curso_aca_ini egresado
            4464         Grado en Criminologia y Seguridad           2012        S
            4464 Grado en Gestión y Administración Pública           2018        N

Alumno 6232:
 per_id_ficticio                             titulacion  curso_aca_ini egresado
            6232                       Grado en Derecho           2010        N
            6232                       Grado en Derecho           2010

In [2]:
# Ver qué titulacion tiene cada alumno múltiple en df_eda_final
for pid_idx, pid in enumerate(multiples[:5]):
    # Necesitamos encontrar la posición de este alumno en df_eda_final
    # df_eda_final no tiene per_id_ficticio directamente
    # Pero df_alumno sí — buscar la primera fila de este alumno en df_alumno
    filas_alumno = df_alumno[df_alumno['per_id_ficticio'] == pid].sort_values('curso_aca_ini')
    
    # Ver qué hay en df_eda_final para las posiciones correspondientes
    # Primero: ¿cuántas veces aparece este alumno en df_eda_final por titulacion?
    tits_eda = df_eda[df_eda['titulacion'].isin(filas_alumno['titulacion'].unique())]
    
    print(f"\nAlumno {pid} — titulaciones en df_alumno: {filas_alumno['titulacion'].unique().tolist()}")
    print(f"  target en df_alumno egresado: {filas_alumno[['titulacion','egresado','curso_aca_ini']].drop_duplicates().values.tolist()}")


Alumno 3672 — titulaciones en df_alumno: ['Grado en Ingeniería en Diseño Industrial y Desarrollo de Productos', 'Grado en Arquitectura Técnica']
  target en df_alumno egresado: [['Grado en Ingeniería en Diseño Industrial y Desarrollo de Productos', 'N', 2014], ['Grado en Arquitectura Técnica', 'N', 2015]]

Alumno 4464 — titulaciones en df_alumno: ['Grado en Criminologia y Seguridad', 'Grado en Gestión y Administración Pública']
  target en df_alumno egresado: [['Grado en Criminologia y Seguridad', 'S', 2012], ['Grado en Gestión y Administración Pública', 'N', 2018]]

Alumno 6232 — titulaciones en df_alumno: ['Grado en Derecho', 'Grado en Maestro en Educación Primaria']
  target en df_alumno egresado: [['Grado en Derecho', 'N', 2010], ['Grado en Maestro en Educación Primaria', 'S', 2011], ['Grado en Maestro en Educación Primaria', 'N', 2011]]

Alumno 9102 — titulaciones en df_alumno: ['Grado en Criminologia y Seguridad', 'Grado en Derecho']
  target en df_alumno egresado: [['Grado en C

In [3]:
# ¿Cuántas filas tiene df_eda_final? ¿Coincide con alumnos únicos?
print(f"Filas df_eda_final: {len(df_eda)}")
print(f"Alumnos únicos df_alumno: {df_alumno['per_id_ficticio'].nunique()}")
print(f"¿Coinciden? {len(df_eda) == df_alumno['per_id_ficticio'].nunique()}")
print(f"\nAbandonos en df_eda_final: {df_eda['abandono'].value_counts().to_dict()}")

Filas df_eda_final: 33621
Alumnos únicos df_alumno: 30872
¿Coinciden? False

Abandonos en df_eda_final: {0: 23788, 1: 9833}


In [4]:
# ¿Tiene per_id_ficticio df_eda_final? 
# Si no, necesitamos reconstruir qué filas corresponden a qué alumno

# Primero: ver si el notebook de Fase 3 dejó alguna pista
# Cargar df_exp_automl_D_strict si existe
ruta_automl = Path(r'C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl')

# Ver todos los parquets disponibles
for f in sorted(ruta_automl.glob('*.parquet')):
    df_tmp = pd.read_parquet(f)
    tiene_id = 'per_id_ficticio' in df_tmp.columns
    print(f"{f.name}: {df_tmp.shape} — per_id_ficticio: {tiene_id}")

automl_top_modelos.parquet: (10, 13) — per_id_ficticio: False
dataset_final_tfm.parquet: (33621, 20) — per_id_ficticio: False
df_exp_automl_C.parquet: (33621, 36) — per_id_ficticio: False
df_exp_automl_D.parquet: (33621, 22) — per_id_ficticio: False
df_exp_automl_D_strict.parquet: (33621, 20) — per_id_ficticio: False
quick_baseline_casoD.parquet: (10, 10) — per_id_ficticio: False
ranking_final_fase4.parquet: (19, 11) — per_id_ficticio: False
results_autogluon.parquet: (28, 13) — per_id_ficticio: False
results_baselines.parquet: (12, 13) — per_id_ficticio: False
results_h2o.parquet: (44, 13) — per_id_ficticio: False
results_lazypredict.parquet: (54, 13) — per_id_ficticio: False
results_pycaret.parquet: (30, 13) — per_id_ficticio: False


In [5]:
# Ver qué hay en df_eda_final exactamente
# ¿Tiene titulacion como columna con repeticiones?
print("Filas por titulacion (top 10):")
print(df_eda['titulacion'].value_counts().head(10))

print(f"\n¿Tiene per_id_ficticio? {'per_id_ficticio' in df_eda.columns}")
print(f"Titulaciones únicas en df_eda: {df_eda['titulacion'].nunique()}")

Filas por titulacion (top 10):
titulacion
Grado en Administración de Empresas                                   2334
Grado en Maestro en Educación Primaria                                2165
Grado en Psicología                                                   2033
Grado en Derecho                                                      1919
Grado en Finanzas y Contabilidad                                      1778
Grado en Maestro en Educación Infantil                                1486
Grado en Ingeniería en Diseño Industrial y Desarrollo de Productos    1361
Grado en Publicidad y Relaciones Públicas                             1247
Grado en Traducción e Interpretación                                  1170
Grado en Ingeniería Informática                                       1134
Name: count, dtype: int64

¿Tiene per_id_ficticio? False
Titulaciones únicas en df_eda: 40


In [6]:
ruta_model = Path(r'C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\05_modelado')
X_test = pd.read_parquet(ruta_model / 'X_test_prep.parquet')

# ¿Los índices de X_test son posiciones en df_eda_final?
print(f"X_test shape: {X_test.shape}")
print(f"X_test index min/max: {X_test.index.min()} / {X_test.index.max()}")
print(f"df_eda_final shape: {df_eda.shape}")

# Extraer meta directamente
meta_test = df_eda.iloc[X_test.index][['titulacion', 'rama', 'sexo', 
                                        'pais_nombre', 'provincia', 
                                        'via_acceso', 'abandono']].copy()
meta_test.index = X_test.index

print(f"\nmeta_test shape: {meta_test.shape}")
print(f"Titulaciones en test: {meta_test['titulacion'].nunique()}")
print(f"Mínimo casos por titulación:")
print(meta_test['titulacion'].value_counts().tail(5))
print(f"\nPrimeras 3 filas:")
print(meta_test.head(3))

X_test shape: (6725, 19)
X_test index min/max: 16 / 33619
df_eda_final shape: (33621, 21)

meta_test shape: (6725, 7)
Titulaciones en test: 40
Mínimo casos por titulación:
titulacion
Grado en Ingeniería de la Edificación                                41
Doble Grado en Administración y Dirección de Empresas y Derecho,     22
Grado en Ingeniería Agroalimentaria y del Medio Rural (Plan 2018)    17
Grado en Criminologia y Seguridad  (Plan 2020)                       16
Grado en Arquitectura Técnica (Plan 2020)                             8
Name: count, dtype: int64

Primeras 3 filas:
                             titulacion rama    sexo pais_nombre provincia  \
16823  Grado en Finanzas y Contabilidad   SO   Mujer      España  Castelló   
4904                Grado en Psicología   SA  Hombre      España  Castelló   
11906                 Grado en Medicina   SA   Mujer      España  Castelló   

                           via_acceso  abandono  
16823  Pruebas acceso Bachiller Logse         0  